# Reward Graph Exploration

Explore shortest paths on Cogniland maps using an HP-drain cost graph.

- **Edge cost** = `alpha * hp_drain[terrain(dest)]`, except edges into berry tiles cost **0**
- **`alpha`** scales all costs — lower alpha makes the agent more willing to cross expensive terrain

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../scripts"))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra as scipy_dijkstra

from cogniland.envs.tile_effects import TileEffects, drain_for
import generate_maps as gm

# ── Load maps ──────────────────────────────────────────────────────────────
dataset = torch.load("../data/maps/val.pt", map_location="cpu", weights_only=False)
rgb_all = dataset["rgb"].numpy()        # [N, 128, 128, 3]
heightmap_all = dataset["heightmap"].numpy()  # [N, 128, 128]
tidx_all = dataset["terrain_idx"].numpy()     # [N, 128, 128]  int8, -1 = deadly
berry_all = dataset["berry_mask"].numpy()     # [N, 128, 128]  bool
biomes = dataset["biomes"]

fx = TileEffects()
TERRAIN_NAMES = gm.TERRAIN_NAMES  # 9 terrain classes
HP_DRAIN = np.array([fx.hp_drain[t] for t in TERRAIN_NAMES], dtype=np.float32)

MAP_IDS = [0, 4, 8]  # one per biome-ish
MAP_SIZE = 128

print(f"Loaded {len(biomes)} val maps, biomes: {biomes}")
print(f"Terrain HP drains: {dict(zip(TERRAIN_NAMES, HP_DRAIN))}")

In [ ]:
# ── Visualize selected maps ────────────────────────────────────────────────

fig, axes = plt.subplots(1, len(MAP_IDS), figsize=(5 * len(MAP_IDS), 5))
for ax, mid in zip(axes, MAP_IDS):
    ax.imshow(rgb_all[mid])
    ax.set_title(f"Map {mid} — {biomes[mid]}", fontsize=12)
    ax.axis("off")

# Terrain drain legend
drain_text = "  |  ".join(f"{t}: {int(HP_DRAIN[i])}" for i, t in enumerate(TERRAIN_NAMES))
fig.text(0.5, 0.02, f"HP drain per step:  {drain_text}", ha="center", fontsize=8, color="gray")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## HP-drain cost graph

Build a 4-connected grid graph where:
- Edge cost into cell `(r, c)` = `alpha * hp_drain[terrain(r, c)]`
- Edge cost into a **berry tile** = **0** (free healing)
- Deadly border cells (`terrain_idx == -1`) have no edges

`alpha` scales all costs uniformly — play with it to see how the shortest path changes.

In [ ]:
def build_drain_graph(tidx: np.ndarray, berry_mask: np.ndarray, alpha: float = 1.0):
    """Build a sparse graph where edge cost = alpha * hp_drain[dest terrain].

    Berry tiles have cost 0 (free healing).
    Deadly cells (tidx == -1) are disconnected.

    Args:
        tidx: int8 [H, W] terrain index per cell (-1 = deadly)
        berry_mask: bool [H, W] berry locations
        alpha: cost multiplier (>0)

    Returns:
        csr_matrix [H*W, H*W] adjacency matrix
    """
    H, W = tidx.shape
    N = H * W

    # Per-cell cost: alpha * drain, but 0 for berries, inf for deadly
    cell_cost = np.full((H, W), np.inf, dtype=np.float64)
    valid = tidx >= 0
    cell_cost[valid] = alpha * HP_DRAIN[tidx[valid]]
    cell_cost[berry_mask & valid] = 0.0

    # Horizontal edges (left-right)
    r_h, c_h = np.mgrid[0:H, 0:W-1]
    src_h = (r_h * W + c_h).ravel()
    dst_h = (r_h * W + c_h + 1).ravel()
    cost_right = cell_cost[r_h, c_h + 1].ravel()  # cost of entering right neighbor
    cost_left = cell_cost[r_h, c_h].ravel()        # cost of entering left neighbor

    # Vertical edges (up-down)
    r_v, c_v = np.mgrid[0:H-1, 0:W]
    src_v = (r_v * W + c_v).ravel()
    dst_v = ((r_v + 1) * W + c_v).ravel()
    cost_down = cell_cost[r_v + 1, c_v].ravel()
    cost_up = cell_cost[r_v, c_v].ravel()

    # Assemble: both directions for each edge pair
    all_src = np.concatenate([src_h, dst_h, src_v, dst_v])
    all_dst = np.concatenate([dst_h, src_h, dst_v, src_v])
    all_cost = np.concatenate([cost_right, cost_left, cost_down, cost_up])

    # Remove infinite edges (deadly cells)
    finite = np.isfinite(all_cost)
    return csr_matrix((all_cost[finite], (all_src[finite], all_dst[finite])), shape=(N, N))


DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def trace_path(ctg: np.ndarray, spawn: tuple, target: tuple, max_steps: int = 5000):
    """Greedily follow the cost-to-go gradient from spawn to target."""
    H, W = ctg.shape
    path = [spawn]
    r, c = spawn
    for _ in range(max_steps):
        if (r, c) == target:
            break
        best_cost, best_next = ctg[r, c], None
        for dr, dc in DELTAS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and ctg[nr, nc] < best_cost:
                best_cost = ctg[nr, nc]
                best_next = (nr, nc)
        if best_next is None:
            break
        r, c = best_next
        path.append((r, c))
    return path


def sample_spawn_target(tidx, seed=42, min_manhattan=60):
    """Sample spawn and target on land with minimum Manhattan distance."""
    land = np.argwhere(tidx > 2)  # above water
    rng = np.random.RandomState(seed)
    for _ in range(500):
        i, j = rng.randint(len(land)), rng.randint(len(land))
        s, t = tuple(land[i]), tuple(land[j])
        if abs(s[0] - t[0]) + abs(s[1] - t[1]) >= min_manhattan:
            return s, t
    return tuple(land[0]), tuple(land[-1])


# Sample spawn/target for each map
spawns, targets = [], []
for mid in MAP_IDS:
    s, t = sample_spawn_target(tidx_all[mid], seed=mid + 100)
    spawns.append(s)
    targets.append(t)
    print(f"Map {mid}: spawn={s} ({TERRAIN_NAMES[tidx_all[mid][s]]}), "
          f"target={t} ({TERRAIN_NAMES[tidx_all[mid][t]]})")

## Shortest paths for different alpha values

Each row shows one map. Columns show `alpha = 0.1, 0.5, 1.0, 2.0` — how the shortest path changes as terrain costs are scaled up.

In [ ]:
alphas = [0.1, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(len(MAP_IDS), len(alphas), figsize=(5 * len(alphas), 5 * len(MAP_IDS)))

for row, (mid, spawn, target) in enumerate(zip(MAP_IDS, spawns, targets)):
    tidx = tidx_all[mid]
    berries = berry_all[mid]
    H, W = tidx.shape

    for col, alpha in enumerate(alphas):
        ax = axes[row, col]

        # Build graph and run Dijkstra from target (reverse)
        graph = build_drain_graph(tidx, berries, alpha=alpha)
        target_flat = target[0] * W + target[1]
        dist = scipy_dijkstra(graph.T, directed=True, indices=target_flat)
        ctg = dist.reshape(H, W)

        # Trace shortest path
        path = trace_path(ctg, spawn, target)
        total_cost = ctg[spawn]

        # Plot
        ax.imshow(rgb_all[mid])
        if len(path) > 1:
            cols = [c for r, c in path]
            rows = [r for r, c in path]
            ax.plot(cols, rows, "r-", linewidth=1.5, alpha=0.9)
        ax.plot(spawn[1], spawn[0], "o", color="lime", markersize=7, markeredgecolor="k")
        ax.plot(target[1], target[0], "*", color="gold", markersize=12, markeredgecolor="k")
        ax.set_title(f"alpha={alpha}  |  {len(path)} steps  |  cost={total_cost:.0f}", fontsize=10)
        ax.axis("off")

    axes[row, 0].set_ylabel(f"Map {mid}\n{biomes[mid]}", fontsize=11, rotation=0, labelpad=60, va="center")

plt.tight_layout()
plt.show()

## Cost-to-go heatmap (alpha = 1.0)

Dijkstra distance from every cell to the target. Dark = close, bright = far. Black = unreachable.

In [ ]:
alpha = 1.0

fig, axes = plt.subplots(2, len(MAP_IDS), figsize=(5 * len(MAP_IDS), 10))

for i, (mid, spawn, target) in enumerate(zip(MAP_IDS, spawns, targets)):
    tidx = tidx_all[mid]
    berries = berry_all[mid]
    H, W = tidx.shape

    graph = build_drain_graph(tidx, berries, alpha=alpha)
    target_flat = target[0] * W + target[1]
    dist = scipy_dijkstra(graph.T, directed=True, indices=target_flat)
    ctg = dist.reshape(H, W)

    path = trace_path(ctg, spawn, target)

    # Top row: map + path
    ax = axes[0, i]
    ax.imshow(rgb_all[mid])
    if len(path) > 1:
        ax.plot([c for _, c in path], [r for r, _ in path], "r-", linewidth=1.5)
    ax.plot(spawn[1], spawn[0], "o", color="lime", markersize=7, markeredgecolor="k")
    ax.plot(target[1], target[0], "*", color="gold", markersize=12, markeredgecolor="k")
    ax.set_title(f"Map {mid} ({biomes[mid]}) — {len(path)} steps", fontsize=11)
    ax.axis("off")

    # Bottom row: cost-to-go heatmap
    ax2 = axes[1, i]
    ctg_display = np.where(np.isinf(ctg), np.nan, ctg)
    im = ax2.imshow(ctg_display, cmap="inferno_r", origin="upper",
                    vmin=np.nanmin(ctg_display), vmax=np.nanpercentile(ctg_display, 98))
    ax2.plot(target[1], target[0], "*", color="cyan", markersize=12, markeredgecolor="k")
    ax2.set_title(f"Cost-to-go (alpha={alpha})", fontsize=11)
    ax2.axis("off")
    plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04, label="Dijkstra cost")

plt.tight_layout()
plt.show()